In [158]:
import pickle
import pandas as pd
import numpy as np

In [159]:
df = pickle.load(open("dataset_level1.pkl","rb"))

In [160]:
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,2,Sri Lanka,Pakistan,0.1,1,0,Abu Dhabi,Sheikh Zayed Stadium
1,2,Sri Lanka,Pakistan,0.2,1,0,Abu Dhabi,Sheikh Zayed Stadium
2,2,Sri Lanka,Pakistan,0.3,2,0,Abu Dhabi,Sheikh Zayed Stadium
3,2,Sri Lanka,Pakistan,0.4,0,0,Abu Dhabi,Sheikh Zayed Stadium
4,2,Sri Lanka,Pakistan,0.5,0,0,Abu Dhabi,Sheikh Zayed Stadium


In [161]:
df.shape

(63612, 8)

In [162]:
df.isnull().sum()

match_id               0
batting_team           0
bowling_team           0
ball                   0
runs                   0
player_dismissed       0
city                8671
venue                  0
dtype: int64

In [163]:
df[df["city"].isnull()]["venue"].value_counts()

venue
Dubai International Cricket Stadium        3092
Pallekele International Cricket Stadium    2066
Melbourne Cricket Ground                   1453
Sydney Cricket Ground                       749
Adelaide Oval                               498
Harare Sports Club                          372
Sharjah Cricket Stadium                     249
Sylhet International Cricket Stadium        128
Carrara Oval                                 64
Name: count, dtype: int64

In [164]:
cities = np.where(df["city"].isnull(),df["venue"].str.split().apply(lambda x:x[0]),df["city"])

In [165]:
df["city"] = cities

In [166]:
df.isnull().sum()

match_id            0
batting_team        0
bowling_team        0
ball                0
runs                0
player_dismissed    0
city                0
venue               0
dtype: int64

In [167]:
df.drop(columns=["venue"],inplace=True)

In [168]:
eligible_cities = df.city.value_counts()[df.city.value_counts() > 600].index.tolist()

In [169]:
df = df[df["city"].isin(eligible_cities)]

In [170]:
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city
0,2,Sri Lanka,Pakistan,0.1,1,0,Abu Dhabi
1,2,Sri Lanka,Pakistan,0.2,1,0,Abu Dhabi
2,2,Sri Lanka,Pakistan,0.3,2,0,Abu Dhabi
3,2,Sri Lanka,Pakistan,0.4,0,0,Abu Dhabi
4,2,Sri Lanka,Pakistan,0.5,0,0,Abu Dhabi


In [171]:
df.dtypes

match_id              int64
batting_team            str
bowling_team            str
ball                float64
runs                  int64
player_dismissed        str
city                    str
dtype: object

In [172]:
df["current_score"] = df.groupby("match_id")["runs"].cumsum()

In [173]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score
0,2,Sri Lanka,Pakistan,0.1,1,0,Abu Dhabi,1
1,2,Sri Lanka,Pakistan,0.2,1,0,Abu Dhabi,2
2,2,Sri Lanka,Pakistan,0.3,2,0,Abu Dhabi,4
3,2,Sri Lanka,Pakistan,0.4,0,0,Abu Dhabi,4
4,2,Sri Lanka,Pakistan,0.5,0,0,Abu Dhabi,4
...,...,...,...,...,...,...,...,...
115288,964,New Zealand,Pakistan,19.3,1,0,Barbados,126
115289,964,New Zealand,Pakistan,19.4,0,0,Barbados,126
115290,964,New Zealand,Pakistan,19.5,0,0,Barbados,126
115291,964,New Zealand,Pakistan,19.6,1,DL Vettori,Barbados,127


In [174]:
df["over"] = df["ball"].apply(lambda x : str(x).split(".")[0])
df["ball_no"] = df["ball"].apply(lambda x : str(x).split(".")[1])

In [175]:
df["ball_bowled"] = (df["over"].astype("int")*6) + df["ball_no"].astype("int")

In [176]:
df["balls_left"] = 120 - df["ball_bowled"]
df["balls_left"] = df["balls_left"].apply(lambda x:0 if x<0 else x)

In [177]:
df["balls_left"].describe()

count    49241.000000
mean        60.101887
std         34.602862
min          0.000000
25%         30.000000
50%         60.000000
75%         90.000000
max        119.000000
Name: balls_left, dtype: float64

In [178]:
df["player_dismissed"] = df["player_dismissed"].apply(lambda x:0 if x == "0" else 1)
df["player_dismissed"] = df["player_dismissed"].astype("int")
df["player_dismissed"] = df.groupby("match_id")["player_dismissed"].cumsum()
df["wickets_left"] = 10 - df["player_dismissed"]

In [179]:
df["wickets_left"].describe()

count    49241.000000
mean         7.388436
std          2.149355
min          0.000000
25%          6.000000
50%          8.000000
75%          9.000000
max         10.000000
Name: wickets_left, dtype: float64

In [180]:
df["crr"] = (df["current_score"]*6) / df["ball_bowled"]

In [181]:
df["crr"].describe()

count    49241.000000
mean         7.340148
std          2.355854
min          0.000000
25%          6.000000
50%          7.343284
75%          8.636364
max         84.000000
Name: crr, dtype: float64

In [182]:
groups = df.groupby("match_id")
match_ids = df["match_id"].unique()
last_five = []
for id in match_ids:
    last_five.extend(groups.get_group(id).rolling(window = 30)["runs"].sum().values.tolist())

In [183]:
df["last_five"] = last_five

In [184]:
df.columns

Index(['match_id', 'batting_team', 'bowling_team', 'ball', 'runs',
       'player_dismissed', 'city', 'current_score', 'over', 'ball_no',
       'ball_bowled', 'balls_left', 'wickets_left', 'crr', 'last_five'],
      dtype='str')

In [185]:
final_df = df.groupby("match_id")["runs"].sum().reset_index().merge(df,on="match_id")

In [186]:
final_df=final_df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five','runs_x']]

In [187]:
final_df.dropna(inplace=True)

In [188]:
final_df.isnull().sum()

batting_team     0
bowling_team     0
city             0
current_score    0
balls_left       0
wickets_left     0
crr              0
last_five        0
runs_x           0
dtype: int64

In [189]:
final_df = final_df.sample(final_df.shape[0])

In [190]:
final_df

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five,runs_x
43876,South Africa,Sri Lanka,Colombo,48,83,7,7.783784,34.0,98
8654,Sri Lanka,Australia,Cape Town,69,40,3,5.175000,33.0,101
18854,West Indies,Sri Lanka,Colombo,103,42,6,7.923077,30.0,162
5581,Pakistan,Sri Lanka,Johannesburg,179,4,4,9.258621,50.0,189
37975,Pakistan,Sri Lanka,Colombo,86,47,7,7.068493,35.0,175
...,...,...,...,...,...,...,...,...,...
5088,India,New Zealand,Mount Maunganui,162,0,6,8.100000,45.0,163
25034,England,Pakistan,Dubai,105,29,9,6.923077,38.0,148
5829,Pakistan,South Africa,Abu Dhabi,115,6,4,6.052632,41.0,120
11316,Australia,Sri Lanka,Barbados,67,56,6,6.281250,35.0,168


In [191]:
x = final_df.drop(columns=["runs_x"])
y = final_df["runs_x"]

In [192]:
from sklearn.model_selection import train_test_split

In [193]:
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=1,test_size=0.2)

In [194]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [195]:
trf = ColumnTransformer([
    ("trf",OneHotEncoder(sparse_output=False,drop="first"),["batting_team","bowling_team","city"])
],remainder="passthrough")

In [196]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score,mean_absolute_error

In [197]:
models = {
    "LinearRegression" : LinearRegression(),
    "RandomForestRegressor" : RandomForestRegressor(n_estimators=100,random_state=42),
    "ExtraTreesRegressor" : ExtraTreesRegressor(n_estimators=100,random_state=42),
    "XGBRegressor" : XGBRegressor(n_estimators = 1000,learning_rate = 0.2,max_depth = 12,random_state=1)
}
for name,model in models.items():
    pipe = Pipeline([
        ("step1",trf),
        ("step2",model)
    ])
    pipe.fit(x_train,y_train)
    y_pred = pipe.predict(x_test)
    print(name)
    print("R2Score: ",r2_score(y_test,y_pred))
    print("MAE: ",mean_absolute_error(y_test,y_pred))
    print()

LinearRegression
R2Score:  0.7040177567413397
MAE:  13.117264035186517

RandomForestRegressor
R2Score:  0.978837674593659
MAE:  2.103636563583045

ExtraTreesRegressor
R2Score:  0.9924770234074014
MAE:  0.8734097573980274

XGBRegressor
R2Score:  0.986690104007721
MAE:  1.6992220878601074



In [198]:
pipe = Pipeline([
        ("step1",trf),
        ("step2",ExtraTreesRegressor(n_estimators=100,random_state=42))
    ])
pipe.fit(x_train,y_train)
y_pred = pipe.predict(x_test)

In [199]:
pickle.dump(pipe,open("pipe.pkl","wb"))

In [200]:
# Select a row around 10 overs (60 balls left)
test_row = final_df[
    final_df["balls_left"].between(55, 65)
].sample(1, random_state=42).iloc[0]

overs_done = (120 - test_row["balls_left"]) / 6
wickets_lost = 10 - test_row["wickets_left"]

print("===== ENTER IN STREAMLIT =====")
print("Batting Team:", test_row["batting_team"])
print("Bowling Team:", test_row["bowling_team"])
print("City:", test_row["city"])
print("Current Score:", test_row["current_score"])
print("Overs Done:", overs_done)
print("Wicket:", wickets_lost)
print("Last Five:", test_row["last_five"])

print("\n===== ACTUAL FINAL SCORE =====")
print("Actual Final Score:", test_row["runs_x"])

===== ENTER IN STREAMLIT =====
Batting Team: Sri Lanka
Bowling Team: Pakistan
City: Lahore
Current Score: 88
Overs Done: 10.166666666666666
Wicket: 1
Last Five: 34.0

===== ACTUAL FINAL SCORE =====
Actual Final Score: 165
